In [3]:
import sys
import types
import torchvision.transforms.functional as F

# Buat alias lama
sys.modules['torchvision.transforms.functional_tensor'] = F


In [4]:
import os
import cv2
import torch
from tqdm import tqdm
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer

In [5]:
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())
print("Current device:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))

PyTorch version: 2.2.2+cu121
CUDA available: True
CUDA version: 12.1
Device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 3050 Laptop GPU


In [6]:
# --- STEP 1: MODEL SETUP ---
def setup_super_resolution_model(scale=4):
    """
    Setup Real-ESRGAN super-resolution model with flexible scale (2, 4).
    Args:
        scale (int): The upscale factor (2, 4).
    Returns:
        RealESRGANer: Initialized upsampler model.
    """
    print(f"Setting up Real-ESRGAN model with {scale}x scale...")

    # Pilih model path sesuai scale
    if scale == 2:
        model_path = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth'
    elif scale == 4:
        model_path = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    else:
        raise ValueError("Unsupported scale. Please choose 2, 4, or 8.")

    # Buat model RRDBNet sesuai scale
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=23, num_grow_ch=32, scale=scale)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Model will run on: {device}")

    # Init upsampler
    upsampler = RealESRGANer(
        scale=scale,
        model_path=model_path,
        model=model,
        tile=0,
        tile_pad=10,
        pre_pad=0,
        half=torch.cuda.is_available(),
        device=device
    )

    print("Model setup complete.")
    return upsampler, scale

In [7]:
# --- STEP 2: VIDEO UPSCALING ---
def upscale_video(input_path, output_path, upsampler_model, scale=4):
    """
    Upscale video frames using Real-ESRGAN with chosen scale.
    Args:
        input_path (str): Path ke video input.
        output_path (str): Path untuk simpan video hasil.
        upsampler_model (RealESRGANer): Model super-resolution.
        scale (int): Faktor upscale (2, 4).
    """
    try:
        cap = cv2.VideoCapture(input_path)
        if not cap.isOpened():
            print(f"Error: Could not open input video file at {input_path}")
            return

        fps = cap.get(cv2.CAP_PROP_FPS)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        new_width, new_height = width * scale, height * scale

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (new_width, new_height))

        print("\n" + "="*50)
        print(f"Processing Video: {os.path.basename(input_path)}")
        print(f"  - Original Resolution: {width}x{height}")
        print(f"  - Target Resolution:   {new_width}x{new_height}")
        print(f"  - Scale Factor:        {scale}x")
        print(f"  - Frame Rate (FPS):    {fps:.2f}")
        print(f"  - Total Frames:        {frame_count}")
        print("="*50 + "\n")

        for _ in tqdm(range(frame_count), desc="Upscaling Frames", unit="frame"):
            ret, frame = cap.read()
            if not ret:
                break

            output_frame, _ = upsampler_model.enhance(frame, outscale=scale)
            out.write(output_frame)

        cap.release()
        out.release()

        print("\n" + "="*50)
        print("SUCCESS!")
        print(f"Video has been upscaled and saved to: {output_path}")
        print("="*50)

    except Exception as e:
        print(f"\nAn error occurred during video processing: {e}")

In [8]:
# --- STEP 4: DIRECTORY PROCESSING FUNCTION ---
# This new function iterates through all subdirectories of a root folder,
# finds all video files, and processes them.

def process_directory(input_dir, output_dir, upsampler_model, scale=4):
    """
    Recursively finds all video files in the input_dir, upscales them,
    and saves them to the output_dir, preserving the directory structure.
    """
    video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.flv', '.wmv')

    # Walk through the directory tree
    for root, _, files in os.walk(input_dir):
        for filename in files:
            # Check if the file is a video
            if filename.lower().endswith(video_extensions):
                input_path = os.path.join(root, filename)

                # Create the corresponding output directory structure
                # This mirrors the folder hierarchy from the input directory
                relative_path = os.path.relpath(root, input_dir)
                output_subdir = os.path.join(output_dir, relative_path)
                os.makedirs(output_subdir, exist_ok=True)

                # Define the output file path with an '_upscaled' suffix
                base, ext = os.path.splitext(filename)
                output_path = os.path.join(output_subdir, f"{base}_upscaled{ext}")

                # A helpful feature: skip if the upscaled file already exists
                if os.path.exists(output_path):
                    print(f"Skipping already processed file: {output_path}")
                    continue

                # Process the individual video file
                upscale_video(input_path, output_path, upsampler_model, scale=scale)

In [ ]:
input_root_dir = '../testing_preprocess'
output_root_dir = '../testing_output'

if os.path.isdir(input_root_dir):
    # 1. Set up the super-resolution model
    upsampler = setup_super_resolution_model(scale=4)

    # 2. Process all videos in the specified directory
    process_directory(input_root_dir, output_root_dir, upsampler, )
else:
    print(f"Error: Input directory not found at '{input_root_dir}'.")
    print("Please make sure the path is correct.")

Setting up Real-ESRGAN model with 4x scale...
Model will run on: cuda
Model setup complete.


: 